# Graph Neural Network-Based Dynamic User Pairing and Slot Assignment for Hybrid NOMA Networks
## Google Colab Implementation (Version C)
**Academic B.Tech / B.E. Final-Year Capstone Project**

This notebook implements an end-to-end simulation, combinatorial oracle, graph neural network training,
multi-algorithm benchmarking, and publication-quality figure generation.

### Cell 1: Install Dependencies
Installs required Python libraries (quietly if already installed).

In [ ]:
# Cell 1: Install dependencies
!pip install -q numpy scipy torch matplotlib pandas pyyaml networkx

### Cell 2: Import Libraries
Imports standard scientific computing, wireless analysis, graph processing, and PyTorch packages.

In [ ]:
# Cell 2: Import libraries
import os
import sys
import time
import json
import random
import itertools
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Set, Any

import numpy as np
import pandas as pd
import networkx as nx

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

### Cell 3: Set Random Seeds and Configuration
Configures network geometry, physical layer parameters, GNN hyperparameters, and random seeds.

In [ ]:
# Cell 3: Set random seeds and configuration
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Execution Device: {DEVICE}")

@dataclass
class ColabConfig:
    num_users: int = 10
    cell_radius_min: float = 20.0       # meters
    cell_radius_max: float = 500.0      # meters
    carrier_frequency: float = 2.4e9    # Hz
    total_bandwidth: float = 10.0e6     # Hz (10 MHz)
    num_slots: int = 5                  # Resource slots
    total_transmit_power_w: float = 1.0 # Watts (30 dBm)
    noise_spectral_density_dbm: float = -174.0 # dBm/Hz
    noise_figure_db: float = 5.0        # dB
    path_loss_exponent: float = 3.5
    reference_distance: float = 1.0
    reference_path_loss_db: float = 38.4
    alpha_weak: float = 0.75
    alpha_strong: float = 0.25
    sic_residual_factor: float = 0.01
    gain_ratio_threshold: float = 1.5
    hidden_dim: int = 64
    num_gnn_layers: int = 3
    dropout: float = 0.1
    learning_rate: float = 0.001
    weight_decay: float = 1.0e-4
    epochs: int = 40
    fairness_weight: float = 0.2
    min_rate_bps: float = 1.0e5

cfg = ColabConfig()
print("Configuration successfully loaded.")

### Cell 4: Define Channel and Network Simulation
Generates 2D user coordinates, log-distance path loss, complex Rayleigh fading, and noise power.

In [ ]:
# Cell 4: Define channel and network simulation
@dataclass
class UserChannelState:
    user_id: int
    x: float
    y: float
    distance: float
    path_loss_db: float
    path_loss_linear: float
    fading_complex: complex
    channel_complex: complex
    channel_gain: float
    channel_gain_db: float
    snr_linear: float
    snr_db: float
    min_rate_req_bps: float

class WirelessNetwork:
    def __init__(self, config: ColabConfig, seed: Optional[int] = None):
        self.cfg = config
        self.rng = np.random.default_rng(seed)
        n0_w_hz = 10.0 ** ((self.cfg.noise_spectral_density_dbm - 30.0) / 10.0)
        nf_linear = 10.0 ** (self.cfg.noise_figure_db / 10.0)
        self.bandwidth_per_slot = self.cfg.total_bandwidth / self.cfg.num_slots
        self.noise_power_per_slot = n0_w_hz * nf_linear * self.bandwidth_per_slot
        self.power_per_slot = self.cfg.total_transmit_power_w / self.cfg.num_slots

    def set_seed(self, seed: int) -> None:
        self.rng = np.random.default_rng(seed)

    def calculate_path_loss_db(self, distance: np.ndarray) -> np.ndarray:
        d_clipped = np.maximum(distance, self.cfg.reference_distance)
        return self.cfg.reference_path_loss_db + 10.0 * self.cfg.path_loss_exponent * np.log10(d_clipped / self.cfg.reference_distance)

    def generate_users(self, num_users: Optional[int] = None) -> List[UserChannelState]:
        n = num_users if num_users is not None else self.cfg.num_users
        r_min_sq, r_max_sq = self.cfg.cell_radius_min**2, self.cfg.cell_radius_max**2
        radii = np.sqrt(self.rng.uniform(0.0, 1.0, size=n) * (r_max_sq - r_min_sq) + r_min_sq)
        angles = self.rng.uniform(0.0, 2.0 * np.pi, size=n)
        x_coords, y_coords = radii * np.cos(angles), radii * np.sin(angles)

        pl_db = self.calculate_path_loss_db(radii)
        pl_linear = 10.0 ** (-pl_db / 10.0)
        sigma = 1.0 / np.sqrt(2.0)
        h_fading = self.rng.normal(0.0, sigma, size=n) + 1j * self.rng.normal(0.0, sigma, size=n)
        h_combined = np.sqrt(pl_linear) * h_fading
        gains = np.abs(h_combined) ** 2

        users = []
        for i in range(n):
            gain = float(gains[i])
            gain_db = 10.0 * np.log10(gain) if gain > 1e-30 else -300.0
            snr_lin = float((self.power_per_slot * gain) / self.noise_power_per_slot)
            users.append(UserChannelState(
                user_id=i, x=float(x_coords[i]), y=float(y_coords[i]), distance=float(radii[i]),
                path_loss_db=float(pl_db[i]), path_loss_linear=float(pl_linear[i]),
                fading_complex=complex(h_fading[i]), channel_complex=complex(h_combined[i]),
                channel_gain=gain, channel_gain_db=gain_db, snr_linear=snr_lin,
                snr_db=10.0 * np.log10(snr_lin) if snr_lin > 1e-30 else -300.0,
                min_rate_req_bps=self.cfg.min_rate_bps,
            ))
        return users

print("WirelessNetwork simulation class defined.")

### Cell 5: Define the PD-NOMA Model
Implements superposition transmission, successive interference cancellation (SIC), and SINR / achievable rates.

In [ ]:
# Cell 5: Define the PD-NOMA model
@dataclass
class PDPairResult:
    weak_user_id: int
    strong_user_id: int
    slot_id: int
    sinr_weak: float
    sinr_strong: float
    rate_weak_bps: float
    rate_strong_bps: float
    pair_sum_rate_bps: float
    sic_successful: bool

class PDNOMAEngine:
    def __init__(self, config: ColabConfig):
        self.cfg = config

    def compute_pair_rates(self, u_a: UserChannelState, u_b: UserChannelState, power_w: float, bandwidth_hz: float, noise_power_w: float, slot_id: int = 0) -> PDPairResult:
        weak_u, strong_u = (u_a, u_b) if u_a.channel_gain <= u_b.channel_gain else (u_b, u_a)
        g_w, g_s = float(weak_u.channel_gain), float(strong_u.channel_gain)
        a_w, a_s = self.cfg.alpha_weak, self.cfg.alpha_strong
        beta_sic = self.cfg.sic_residual_factor

        denom_w = a_s * power_w * g_w + noise_power_w
        sinr_w = (a_w * power_w * g_w) / denom_w if denom_w > 0 else 0.0
        rate_w = bandwidth_hz * np.log2(1.0 + sinr_w)

        denom_s = beta_sic * a_w * power_w * g_s + noise_power_w
        sinr_s = (a_s * power_w * g_s) / denom_s if denom_s > 0 else 0.0
        rate_s = bandwidth_hz * np.log2(1.0 + sinr_s)

        return PDPairResult(
            weak_user_id=weak_u.user_id, strong_user_id=strong_u.user_id, slot_id=slot_id,
            sinr_weak=float(sinr_w), sinr_strong=float(sinr_s),
            rate_weak_bps=float(rate_w), rate_strong_bps=float(rate_s),
            pair_sum_rate_bps=float(rate_w + rate_s), sic_successful=True
        )

    def compute_single_user_oma_rate(self, user: UserChannelState, power_w: float, bandwidth_hz: float, noise_power_w: float) -> float:
        snr = (power_w * user.channel_gain) / noise_power_w if noise_power_w > 0 else 0.0
        return float(bandwidth_hz * np.log2(1.0 + snr))

def compute_jains_fairness_index(rates: List[float]) -> float:
    arr = np.array(rates, dtype=float)
    sum_r, sum_sq = float(np.sum(arr)), float(np.sum(arr**2))
    return (sum_r**2) / (len(arr) * sum_sq) if sum_sq > 0 else 1.0

print("PDNOMAEngine defined.")

### Cell 6: Define the SCMA Model
Implements regular factor graph matrix (6 users, 4 resources), complex codebooks, and iterative MPA detector.

In [ ]:
# Cell 6: Define the SCMA model
class SCMAEngine:
    def __init__(self, J: int = 6, K: int = 4, M: int = 4, n_iter: int = 4):
        self.J, self.K, self.M, self.n_iter = J, K, M, n_iter
        self.F = np.array([
            [1, 1, 1, 0, 0, 0],
            [1, 0, 0, 1, 1, 0],
            [0, 1, 0, 1, 0, 1],
            [0, 0, 1, 0, 1, 1],
        ], dtype=int)
        self.V_k = [[j for j in range(self.J) if self.F[k, j] == 1] for k in range(self.K)]
        self.K_j = [[k for k in range(self.K) if self.F[k, j] == 1] for j in range(self.J)]
        self.codebooks = self._generate_codebooks()

    def _generate_codebooks(self) -> np.ndarray:
        cb = np.zeros((self.J, self.M, self.K), dtype=complex)
        qpsk = np.array([np.exp(1j * (np.pi/4.0 + m * np.pi/2.0)) for m in range(self.M)], dtype=complex) / np.sqrt(2.0)
        for j in range(self.J):
            res = self.K_j[j]
            rot = np.exp(1j * (j * np.pi / self.J))
            for m in range(self.M):
                pt = qpsk[m] * rot
                cb[j, m, res[0]] = np.real(pt)
                cb[j, m, res[1]] = 1j * np.imag(pt)
        return cb

    def mpa_detector(self, y: np.ndarray, H: np.ndarray, noise_var: float) -> np.ndarray:
        I_v_to_f = np.ones((self.K, self.J, self.M), dtype=float) / self.M
        I_f_to_v = np.zeros((self.K, self.J, self.M), dtype=float)

        for _ in range(self.n_iter):
            for k in range(self.K):
                for target_j in self.V_k[k]:
                    interfering = [u for u in self.V_k[k] if u != target_j]
                    for m_target in range(self.M):
                        accum = 0.0
                        for combo in itertools.product(range(self.M), repeat=len(interfering)):
                            sig = H[k, target_j] * self.codebooks[target_j, m_target, k]
                            prior = 1.0
                            for idx_u, u_oth in enumerate(interfering):
                                sig += H[k, u_oth] * self.codebooks[u_oth, combo[idx_u], k]
                                prior *= I_v_to_f[k, u_oth, combo[idx_u]]
                            accum += np.exp(-float(np.abs(y[k] - sig)**2) / max(noise_var, 1e-12)) * prior
                        I_f_to_v[k, target_j, m_target] = accum
                    s = np.sum(I_f_to_v[k, target_j, :])
                    if s > 0: I_f_to_v[k, target_j, :] /= s

            for j in range(self.J):
                for k_t in self.K_j[j]:
                    others = [k for k in self.K_j[j] if k != k_t]
                    for m in range(self.M):
                        I_v_to_f[k_t, j, m] = (1.0 / self.M) * np.prod([I_f_to_v[k, j, m] for k in others])
                    s = np.sum(I_v_to_f[k_t, j, :])
                    if s > 0: I_v_to_f[k_t, j, :] /= s

        Q = np.zeros((self.J, self.M), dtype=float)
        for j in range(self.J):
            for m in range(self.M):
                Q[j, m] = (1.0 / self.M) * np.prod([I_f_to_v[k, j, m] for k in self.K_j[j]])
            s = np.sum(Q[j, :])
            if s > 0: Q[j, :] /= s
        return Q

print("SCMAEngine defined.")

### Cell 7: Define Graph Construction
Constructs node features, edge attributes (gain disparity, spatial distance, correlation), and tensors.

In [ ]:
# Cell 7: Define graph construction
@dataclass
class NetworkGraph:
    num_nodes: int
    node_features: torch.Tensor
    edge_index: torch.Tensor
    edge_features: torch.Tensor
    pair_labels: Optional[torch.Tensor] = None
    slot_labels: Optional[torch.Tensor] = None

class GraphBuilder:
    def __init__(self, r_max: float = 500.0):
        self.r_max = r_max

    def build_graph(self, users: List[UserChannelState], pair_matrix: Optional[np.ndarray] = None, slot_array: Optional[np.ndarray] = None) -> NetworkGraph:
        n = len(users)
        coords = np.array([[u.x, u.y] for u in users])
        distances = np.array([u.distance for u in users])
        gains_lin = np.array([u.channel_gain for u in users])
        gains_db = np.array([u.channel_gain_db for u in users])
        req_rates = np.array([u.min_rate_req_bps for u in users])
        h_complex = np.array([u.channel_complex for u in users])

        mean_gain = max(float(np.mean(gains_lin)), 1e-18)
        mean_db, std_db = float(np.mean(gains_db)), max(float(np.std(gains_db)), 1.0)

        node_feats = np.zeros((n, 6), dtype=np.float32)
        node_feats[:, 0] = coords[:, 0] / self.r_max
        node_feats[:, 1] = coords[:, 1] / self.r_max
        node_feats[:, 2] = distances / self.r_max
        node_feats[:, 3] = (gains_db - mean_db) / std_db
        node_feats[:, 4] = gains_lin / mean_gain
        node_feats[:, 5] = req_rates / 1.0e6

        src, dst, edge_attr = [], [], []
        for i in range(n):
            for j in range(n):
                if i != j:
                    src.append(i); dst.append(j)
                    d_ij = np.linalg.norm(coords[i] - coords[j]) / (2.0 * self.r_max)
                    gain_disp = abs(gains_db[i] - gains_db[j]) / 40.0
                    norm_i, norm_j = np.abs(h_complex[i]), np.abs(h_complex[j])
                    corr = float(np.abs(np.conj(h_complex[i]) * h_complex[j]) / (norm_i * norm_j)) if norm_i > 1e-12 and norm_j > 1e-12 else 0.0
                    interf = float((gains_lin[i] + gains_lin[j]) / (2.0 * mean_gain))
                    edge_attr.append([d_ij, gain_disp, corr, interf])

        return NetworkGraph(
            num_nodes=n,
            node_features=torch.tensor(node_feats, dtype=torch.float32),
            edge_index=torch.tensor([src, dst], dtype=torch.long),
            edge_features=torch.tensor(edge_attr, dtype=torch.float32),
            pair_labels=torch.tensor(pair_matrix, dtype=torch.float32) if pair_matrix is not None else None,
            slot_labels=torch.tensor(slot_array, dtype=torch.long) if slot_array is not None else None,
        )

print("GraphBuilder defined.")

### Cell 8: Define Pairing Algorithms
Implements Random, Near-Far Channel Gain, Greedy Sum-Rate, and GNN Affinity matching.

In [ ]:
# Cell 8: Define pairing algorithms
@dataclass
class UserPair:
    user1_id: int
    user2_id: int
    affinity: float = 1.0

@dataclass
class PairingPlan:
    pairs: List[UserPair]
    unpaired_users: List[int]
    method_name: str

    def validate(self, n: int) -> bool:
        seen = set()
        for p in self.pairs:
            if p.user1_id == p.user2_id or p.user1_id in seen or p.user2_id in seen: return False
            seen.add(p.user1_id); seen.add(p.user2_id)
        for u in self.unpaired_users:
            if u in seen: return False
            seen.add(u)
        return len(seen) == n

class UserPairingManager:
    def __init__(self, pd_engine: PDNOMAEngine):
        self.pd_engine = pd_engine

    def random_pairing(self, users: List[UserChannelState], rng: Optional[np.random.Generator] = None) -> PairingPlan:
        if rng is None: rng = np.random.default_rng()
        ids = rng.permutation([u.user_id for u in users]).tolist()
        n = len(ids)
        pairs = [UserPair(ids[2*i], ids[2*i+1], 1.0) for i in range(n // 2)]
        unpaired = [ids[-1]] if n % 2 != 0 else []
        return PairingPlan(pairs, unpaired, "Random Pairing")

    def channel_gain_pairing(self, users: List[UserChannelState], strategy: str = "near_far") -> PairingPlan:
        sorted_ids = [u.user_id for u in sorted(users, key=lambda u: u.channel_gain)]
        n = len(sorted_ids)
        pairs, unpaired = [], []
        if strategy == "near_far":
            for i in range(n // 2):
                pairs.append(UserPair(sorted_ids[i], sorted_ids[n - 1 - i], 1.0))
            if n % 2 != 0: unpaired.append(sorted_ids[n // 2])
        else:
            for i in range(n // 2):
                pairs.append(UserPair(sorted_ids[2*i], sorted_ids[2*i+1], 1.0))
            if n % 2 != 0: unpaired.append(sorted_ids[-1])
        return PairingPlan(pairs, unpaired, f"Channel Gain ({strategy})")

    def greedy_sum_rate_pairing(self, users: List[UserChannelState], p_w: float, bw_hz: float, n0_w: float) -> PairingPlan:
        n = len(users)
        u_dict = {u.user_id: u for u in users}
        active_ids = set(u_dict.keys())
        u_list = list(u_dict.values())
        cands = []
        for i in range(n):
            for j in range(i + 1, n):
                res = self.pd_engine.compute_pair_rates(u_list[i], u_list[j], p_w, bw_hz, n0_w)
                cands.append((res.pair_sum_rate_bps, u_list[i].user_id, u_list[j].user_id))
        cands.sort(key=lambda x: x[0], reverse=True)
        pairs = []
        for rate, u1, u2 in cands:
            if u1 in active_ids and u2 in active_ids:
                pairs.append(UserPair(u1, u2, rate))
                active_ids.remove(u1); active_ids.remove(u2)
        return PairingPlan(pairs, sorted(list(active_ids)), "Greedy Sum-Rate")

    def gnn_affinity_pairing(self, aff_matrix: np.ndarray, users: List[UserChannelState]) -> PairingPlan:
        n = len(users)
        active = set(range(n))
        sym_aff = 0.5 * (aff_matrix + aff_matrix.T)
        np.fill_diagonal(sym_aff, -np.inf)
        edges = [(float(sym_aff[i, j]), i, j) for i in range(n) for j in range(i + 1, n)]
        edges.sort(key=lambda x: x[0], reverse=True)
        pairs = []
        for weight, i, j in edges:
            if i in active and j in active:
                pairs.append(UserPair(i, j, weight))
                active.remove(i); active.remove(j)
        return PairingPlan(pairs, sorted(list(active)), "Proposed GNN-Based NOMA")

print("UserPairingManager defined.")

### Cell 9: Define Slot Assignment
Evaluates dynamic scheduling across orthogonal slots respecting capacity and power budgets.

In [ ]:
# Cell 9: Define slot assignment
@dataclass
class NetworkScheduleResult:
    total_sum_rate_bps: float
    per_user_rates_bps: Dict[int, float]
    jains_fairness: float
    active_slots: int
    slot_utilization: float
    outage_probability: float

class SlotAssignmentEngine:
    def __init__(self, num_slots: int, total_bw: float, total_pwr: float, noise_pwr: float, pd_engine: PDNOMAEngine):
        self.num_slots = num_slots
        self.slot_bandwidth = total_bw / num_slots
        self.slot_power = total_pwr / num_slots
        self.noise_power_per_slot = noise_pwr
        self.pd_engine = pd_engine

    def evaluate_schedule(self, plan: PairingPlan, users: List[UserChannelState]) -> NetworkScheduleResult:
        u_dict = {u.user_id: u for u in users}
        user_rates = {u.user_id: 0.0 for u in users}
        active_slots = 0

        for idx, pair in enumerate(plan.pairs):
            if idx >= self.num_slots: break
            active_slots += 1
            res = self.pd_engine.compute_pair_rates(
                u_dict[pair.user1_id], u_dict[pair.user2_id],
                self.slot_power, self.slot_bandwidth, self.noise_power_per_slot, slot_id=idx
            )
            user_rates[res.weak_user_id] = res.rate_weak_bps
            user_rates[res.strong_user_id] = res.rate_strong_bps

        for idx, u_id in enumerate(plan.unpaired_users):
            if active_slots >= self.num_slots: break
            active_slots += 1
            rate = self.pd_engine.compute_single_user_oma_rate(
                u_dict[u_id], self.slot_power, self.slot_bandwidth, self.noise_power_per_slot
            )
            user_rates[u_id] = rate

        rates = list(user_rates.values())
        outages = sum(1 for u in users if user_rates[u.user_id] < u.min_rate_req_bps)
        return NetworkScheduleResult(
            total_sum_rate_bps=float(np.sum(rates)),
            per_user_rates_bps=user_rates,
            jains_fairness=compute_jains_fairness_index(rates),
            active_slots=active_slots,
            slot_utilization=float(active_slots / self.num_slots),
            outage_probability=float(outages / len(users)) if users else 0.0,
        )

print("SlotAssignmentEngine defined.")

### Cell 10: Define Optimization / Oracle Labels
Generates globally optimal pairing targets via exhaustive combinatorial search for small N.

In [ ]:
# Cell 10: Define optimization/oracle labels
def generate_all_pair_partitions(elements: List[int]) -> List[List[Tuple[int, int]]]:
    if len(elements) < 2: return [[]]
    first, rest = elements[0], elements[1:]
    partitions = []
    for i, second in enumerate(rest):
        pair = (first, second)
        for sub in generate_all_pair_partitions(rest[:i] + rest[i+1:]):
            partitions.append([pair] + sub)
    return partitions

class OptimizationOracle:
    def __init__(self, slot_engine: SlotAssignmentEngine, pd_engine: PDNOMAEngine):
        self.slot_engine = slot_engine
        self.pd_engine = pd_engine

    def solve(self, users: List[UserChannelState]) -> Tuple[PairingPlan, NetworkScheduleResult]:
        n = len(users)
        u_ids = [u.user_id for u in users]
        best_util, best_plan, best_sched = -1e9, None, None

        if n % 2 == 0:
            candidate_partitions = [(p, []) for p in generate_all_pair_partitions(u_ids)]
        else:
            candidate_partitions = []
            for idx in range(n):
                unp = u_ids[idx]
                pool = u_ids[:idx] + u_ids[idx+1:]
                for p in generate_all_pair_partitions(pool):
                    candidate_partitions.append((p, [unp]))

        for partition, unpaired in candidate_partitions:
            plan = PairingPlan([UserPair(p[0], p[1], 1.0) for p in partition], unpaired, "Optimization Oracle")
            sched = self.slot_engine.evaluate_schedule(plan, users)
            util = sched.total_sum_rate_bps + 0.2 * sched.jains_fairness * 1e6
            if util > best_util:
                best_util, best_plan, best_sched = util, plan, sched

        return best_plan, best_sched

    def generate_labels(self, users: List[UserChannelState]) -> Tuple[np.ndarray, np.ndarray, PairingPlan]:
        n = len(users)
        plan, sched = self.solve(users)
        pair_matrix = np.zeros((n, n), dtype=np.float32)
        for p in plan.pairs:
            pair_matrix[p.user1_id, p.user2_id] = 1.0
            pair_matrix[p.user2_id, p.user1_id] = 1.0
        slot_vector = np.zeros(n, dtype=np.int64)
        for idx, p in enumerate(plan.pairs):
            slot_vector[p.user1_id] = idx % self.slot_engine.num_slots
            slot_vector[p.user2_id] = idx % self.slot_engine.num_slots
        return pair_matrix, slot_vector, plan

print("OptimizationOracle defined.")

### Cell 11: Define the GNN Architecture
Native PyTorch message passing network with edge-conditioned convolution and bilinear pairing head.

In [ ]:
# Cell 11: Define the GNN architecture
class EdgeConditionedConvLayer(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__()
        self.msg_mlp = nn.Sequential(
            nn.Linear(node_dim + edge_dim, out_dim), nn.LeakyReLU(0.2), nn.Dropout(dropout), nn.Linear(out_dim, out_dim)
        )
        self.update_mlp = nn.Sequential(
            nn.Linear(node_dim + out_dim, out_dim), nn.LeakyReLU(0.2), nn.Dropout(dropout), nn.Linear(out_dim, out_dim)
        )
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        n = x.size(0)
        src, dst = edge_index[0], edge_index[1]
        msg = self.msg_mlp(torch.cat([x[src], edge_attr], dim=-1))
        agg = torch.zeros((n, msg.size(-1)), device=x.device)
        agg.index_add_(0, dst, msg)
        deg = torch.clamp(torch.zeros((n, 1), device=x.device).index_add_(0, dst, torch.ones((edge_index.size(1), 1), device=x.device)), min=1.0)
        agg = agg / deg
        return self.norm(x + self.update_mlp(torch.cat([x, agg], dim=-1)))

class HybridNOMAGNN(nn.Module):
    def __init__(self, node_in: int = 6, edge_in: int = 4, hidden_dim: int = 64, num_layers: int = 3, num_slots: int = 5, dropout: float = 0.1):
        super().__init__()
        self.node_embed = nn.Sequential(nn.Linear(node_in, hidden_dim), nn.LeakyReLU(0.2), nn.LayerNorm(hidden_dim))
        self.convs = nn.ModuleList([EdgeConditionedConvLayer(hidden_dim, edge_in, hidden_dim, dropout) for _ in range(num_layers)])
        self.pair_head = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim), nn.LeakyReLU(0.2), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LeakyReLU(0.2), nn.Linear(hidden_dim // 2, 1)
        )
        self.slot_head = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU(0.2), nn.Linear(hidden_dim, num_slots))

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        n = x.size(0)
        h = self.node_embed(x)
        for conv in self.convs: h = conv(h, edge_index, edge_attr)

        h_i, h_j = h.unsqueeze(1).expand(-1, n, -1), h.unsqueeze(0).expand(n, -1, -1)
        pair_repr = torch.cat([h_i, h_j, torch.abs(h_i - h_j), h_i * h_j], dim=-1)
        pair_logits = self.pair_head(pair_repr).squeeze(-1)
        sym_pair_logits = 0.5 * (pair_logits + pair_logits.t())
        sym_pair_logits = sym_pair_logits.masked_fill(torch.eye(n, dtype=torch.bool, device=x.device), -1e9)
        return sym_pair_logits, self.slot_head(h)

    def predict_pairing_probabilities(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            pair_logits, _ = self.forward(x, edge_index, edge_attr)
            probs = torch.sigmoid(pair_logits)
            probs.fill_diagonal_(0.0)
            return probs

print("HybridNOMAGNN defined.")

### Cell 12: Generate Training, Validation, and Test Data
Creates synthetic channel realizations with combinatorial oracle supervision.

In [ ]:
# Cell 12: Generate training, validation, and test data
network = WirelessNetwork(cfg, seed=GLOBAL_SEED)
pd_engine = PDNOMAEngine(cfg)
slot_engine = SlotAssignmentEngine(cfg.num_slots, cfg.total_bandwidth, cfg.total_transmit_power_w, network.noise_power_per_slot, pd_engine)
oracle = OptimizationOracle(slot_engine, pd_engine)
graph_builder = GraphBuilder(cfg.cell_radius_max)

print("Generating synthetic graphs...")
train_graphs = []
for i in range(60):
    network.set_seed(100 + i)
    users = network.generate_users(cfg.num_users)
    p_mat, s_vec, _ = oracle.generate_labels(users)
    train_graphs.append(graph_builder.build_graph(users, p_mat, s_vec))

val_graphs = []
for i in range(20):
    network.set_seed(500 + i)
    users = network.generate_users(cfg.num_users)
    p_mat, s_vec, _ = oracle.generate_labels(users)
    val_graphs.append(graph_builder.build_graph(users, p_mat, s_vec))

print(f"Generated {len(train_graphs)} training graphs and {len(val_graphs)} validation graphs.")

### Cell 13: Train the GNN
Trains GNN using Adam optimizer and tracks validation loss across epochs.

In [ ]:
# Cell 13: Train the GNN
model = HybridNOMAGNN(hidden_dim=cfg.hidden_dim, num_layers=cfg.num_gnn_layers, num_slots=cfg.num_slots, dropout=cfg.dropout).to(DEVICE)
optimizer = Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

train_losses, val_losses = [], []
best_val = float("inf")

def compute_loss(g: NetworkGraph) -> torch.Tensor:
    pair_logits, slot_logits = model(g.node_features.to(DEVICE), g.edge_index.to(DEVICE), g.edge_features.to(DEVICE))
    mask = ~torch.eye(g.num_nodes, dtype=torch.bool, device=DEVICE)
    bce = F.binary_cross_entropy_with_logits(pair_logits[mask], g.pair_labels.to(DEVICE)[mask], pos_weight=torch.tensor([float(g.num_nodes - 2)], device=DEVICE))
    slot_ce = F.cross_entropy(slot_logits, g.slot_labels.to(DEVICE))
    return bce + 0.3 * slot_ce

print("Starting GNN Training...")
for ep in range(1, cfg.epochs + 1):
    model.train()
    t_loss = 0.0
    for g in train_graphs:
        optimizer.zero_grad()
        loss = compute_loss(g)
        loss.backward()
        optimizer.step()
        t_loss += float(loss.item())
    t_loss /= len(train_graphs)

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for g in val_graphs:
            v_loss += float(compute_loss(g).item())
    v_loss /= len(val_graphs)
    scheduler.step(v_loss)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    if ep % 5 == 0 or ep == cfg.epochs:
        print(f"Epoch {ep:02d}/{cfg.epochs:02d} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}")

print("Training Completed.")

### Cell 14: Evaluate Baseline Methods
Evaluates OMA, Random NOMA, Near-Far NOMA, Greedy NOMA, and Optimization Oracle on 30 test topologies.

In [ ]:
# Cell 14: Evaluate baseline methods
pairing_mgr = UserPairingManager(pd_engine)
test_instances = 30
methods = ["OMA", "Random NOMA", "Near-Far NOMA", "Greedy NOMA", "Optimization Oracle"]
results_accum = {m: {"sum_rate": [], "fairness": [], "outage": [], "time_ms": []} for m in methods}

for idx in range(test_instances):
    network.set_seed(2000 + idx)
    u_test = network.generate_users(cfg.num_users)
    n = len(u_test)

    # OMA
    t0 = time.perf_counter()
    oma_sched = slot_engine.evaluate_schedule(PairingPlan([], [u.user_id for u in u_test], "OMA"), u_test)
    results_accum["OMA"]["sum_rate"].append(oma_sched.total_sum_rate_bps / 1e6)
    results_accum["OMA"]["fairness"].append(oma_sched.jains_fairness)
    results_accum["OMA"]["outage"].append(oma_sched.outage_probability)
    results_accum["OMA"]["time_ms"].append((time.perf_counter() - t0) * 1000)

    # Random
    t0 = time.perf_counter()
    r_sched = slot_engine.evaluate_schedule(pairing_mgr.random_pairing(u_test), u_test)
    results_accum["Random NOMA"]["sum_rate"].append(r_sched.total_sum_rate_bps / 1e6)
    results_accum["Random NOMA"]["fairness"].append(r_sched.jains_fairness)
    results_accum["Random NOMA"]["outage"].append(r_sched.outage_probability)
    results_accum["Random NOMA"]["time_ms"].append((time.perf_counter() - t0) * 1000)

    # Near-Far
    t0 = time.perf_counter()
    nf_sched = slot_engine.evaluate_schedule(pairing_mgr.channel_gain_pairing(u_test, "near_far"), u_test)
    results_accum["Near-Far NOMA"]["sum_rate"].append(nf_sched.total_sum_rate_bps / 1e6)
    results_accum["Near-Far NOMA"]["fairness"].append(nf_sched.jains_fairness)
    results_accum["Near-Far NOMA"]["outage"].append(nf_sched.outage_probability)
    results_accum["Near-Far NOMA"]["time_ms"].append((time.perf_counter() - t0) * 1000)

    # Greedy
    t0 = time.perf_counter()
    gr_sched = slot_engine.evaluate_schedule(pairing_mgr.greedy_sum_rate_pairing(u_test, slot_engine.slot_power, slot_engine.slot_bandwidth, slot_engine.noise_power_per_slot), u_test)
    results_accum["Greedy NOMA"]["sum_rate"].append(gr_sched.total_sum_rate_bps / 1e6)
    results_accum["Greedy NOMA"]["fairness"].append(gr_sched.jains_fairness)
    results_accum["Greedy NOMA"]["outage"].append(gr_sched.outage_probability)
    results_accum["Greedy NOMA"]["time_ms"].append((time.perf_counter() - t0) * 1000)

    # Oracle
    t0 = time.perf_counter()
    _, o_sched = oracle.solve(u_test)
    results_accum["Optimization Oracle"]["sum_rate"].append(o_sched.total_sum_rate_bps / 1e6)
    results_accum["Optimization Oracle"]["fairness"].append(o_sched.jains_fairness)
    results_accum["Optimization Oracle"]["outage"].append(o_sched.outage_probability)
    results_accum["Optimization Oracle"]["time_ms"].append((time.perf_counter() - t0) * 1000)

print(f"Evaluated 5 baseline methods over {test_instances} test topologies.")

### Cell 15: Evaluate GNN-Based Pairing
Evaluates trained GNN on the exact same test topologies.

In [ ]:
# Cell 15: Evaluate GNN-based pairing
results_accum["Proposed GNN-Based NOMA"] = {"sum_rate": [], "fairness": [], "outage": [], "time_ms": []}
model.eval()

for idx in range(test_instances):
    network.set_seed(2000 + idx)
    u_test = network.generate_users(cfg.num_users)
    t0 = time.perf_counter()
    g_test = graph_builder.build_graph(u_test)
    aff_probs = model.predict_pairing_probabilities(
        g_test.node_features.to(DEVICE), g_test.edge_index.to(DEVICE), g_test.edge_features.to(DEVICE)
    ).cpu().numpy()
    gnn_plan = pairing_mgr.gnn_affinity_pairing(aff_probs, u_test)
    gnn_sched = slot_engine.evaluate_schedule(gnn_plan, u_test)
    lat = (time.perf_counter() - t0) * 1000

    results_accum["Proposed GNN-Based NOMA"]["sum_rate"].append(gnn_sched.total_sum_rate_bps / 1e6)
    results_accum["Proposed GNN-Based NOMA"]["fairness"].append(gnn_sched.jains_fairness)
    results_accum["Proposed GNN-Based NOMA"]["outage"].append(gnn_sched.outage_probability)
    results_accum["Proposed GNN-Based NOMA"]["time_ms"].append(lat)

print("GNN evaluation completed.")

### Cell 16: Compare Results
Formulates a summary benchmark table displaying mean sum-rate, fairness, outage, and latency.

In [ ]:
# Cell 16: Compare results
rows = []
for m, data in results_accum.items():
    rows.append({
        "Method": m,
        "Sum Rate Mean (Mbps)": float(np.mean(data["sum_rate"])),
        "Sum Rate Std (Mbps)": float(np.std(data["sum_rate"])),
        "Jain's Fairness": float(np.mean(data["fairness"])),
        "Outage Probability": float(np.mean(data["outage"])),
        "Inference Time (ms)": float(np.mean(data["time_ms"])),
    })

df_summary = pd.DataFrame(rows)
print("\n" + "="*85)
print("BENCHMARK COMPARISON ACROSS 30 TEST TOPOLOGIES")
print("="*85)
print(df_summary.to_string(index=False))
print("="*85)

### Cell 17: Generate Plots
Visualizes network topology, training convergence curves, sum-rate bar chart, and SCMA BER vs SNR.

In [ ]:
# Cell 17: Generate plots
os.makedirs("results/figures", exist_ok=True)

# 1. Topology Plot
network.set_seed(42)
demo_users = network.generate_users(cfg.num_users)
demo_g = graph_builder.build_graph(demo_users)
aff = model.predict_pairing_probabilities(demo_g.node_features.to(DEVICE), demo_g.edge_index.to(DEVICE), demo_g.edge_features.to(DEVICE)).cpu().numpy()
demo_plan = pairing_mgr.gnn_affinity_pairing(aff, demo_users)

fig, ax = plt.subplots(figsize=(7, 7))
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(cfg.cell_radius_max*np.cos(theta), cfg.cell_radius_max*np.sin(theta), 'k--', alpha=0.5, label="Cell Boundary (500m)")
ax.scatter([0], [0], color="red", marker="^", s=200, label="Base Station", zorder=5)
for u in demo_users:
    ax.scatter(u.x, u.y, color="blue", s=90, zorder=4)
    ax.annotate(f"U{u.user_id}", (u.x+8, u.y+8), fontsize=9)
u_dict = {u.user_id: u for u in demo_users}
for p in demo_plan.pairs:
    ax.plot([u_dict[p.user1_id].x, u_dict[p.user2_id].x], [u_dict[p.user1_id].y, u_dict[p.user2_id].y], 'r-', lw=2, alpha=0.8)
ax.set_title("GNN-Assigned Hybrid NOMA Network Topology")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.legend(); ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.savefig("results/figures/network_topology.png", dpi=300)
plt.show()

# 2. Convergence Curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, cfg.epochs+1), train_losses, 'b-', lw=2, label="Train Loss")
ax.plot(range(1, cfg.epochs+1), val_losses, 'r--', lw=2, label="Val Loss")
ax.set_title("GNN Training & Validation Loss Convergence")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(); ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.savefig("results/figures/training_loss_curves.png", dpi=300)
plt.show()

# 3. Sum Rate & Fairness Benchmark
fig, ax1 = plt.subplots(figsize=(9, 5))
methods_list = df_summary["Method"].tolist()
x = np.arange(len(methods_list))
w = 0.35
ax1.bar(x - w/2, df_summary["Sum Rate Mean (Mbps)"], w, yerr=df_summary["Sum Rate Std (Mbps)"], capsize=4, label="Sum Rate (Mbps)", color="#1f77b4")
ax1.set_ylabel("Sum Rate (Mbps)", color="#1f77b4", fontweight="bold")
ax1.set_xticks(x); ax1.set_xticklabels([m.replace("Orthogonal Multiple Access (OMA)", "OMA").replace("PD-NOMA", "NOMA") for m in methods_list], rotation=20, ha="right")
ax2 = ax1.twinx()
ax2.bar(x + w/2, df_summary["Jain's Fairness"], w, label="Jain's Fairness", color="#2ca02c")
ax2.set_ylabel("Jain's Fairness", color="#2ca02c", fontweight="bold"); ax2.set_ylim(0, 1.1)
plt.title("Comparative Performance: Sum-Rate vs. Jain's Fairness Index", fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/sum_rate_benchmark.png", dpi=300)
plt.show()

# 4. SCMA BER vs SNR Simulation
scma_engine = SCMAEngine()
snrs = [-5, 0, 5, 10, 15, 20]
bers, sers = [], []
for s in snrs:
    res = scma_engine.simulate_transmission(channel_gains=np.ones(6), snr_db=float(s), num_blocks=100)
    bers.append(max(res.bit_error_rate, 1e-4))
    sers.append(max(res.symbol_error_rate, 1e-4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(snrs, sers, 's--', color="#ff7f0e", lw=2, label="Symbol Error Rate (SER)")
ax.semilogy(snrs, bers, 'o-', color="#d62728", lw=2, label="Bit Error Rate (BER)")
ax.set_title("SCMA MPA Detector Error Rate vs. SNR (150% Overloading)")
ax.set_xlabel("SNR (dB)"); ax.set_ylabel("Error Probability (Log Scale)"); ax.set_ylim(5e-5, 1.0); ax.legend(); ax.grid(True, which="both", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.savefig("results/figures/scma_ber_curve.png", dpi=300)
plt.show()

### Cell 18: Save and Download Results
Exports metrics to CSV/JSON and bundles files for Colab download.

In [ ]:
# Cell 18: Save and download results
os.makedirs("results/metrics", exist_ok=True)
df_summary.to_csv("results/metrics/colab_benchmark_results.csv", index=False)

summary_data = {
    "parameters": asdict(cfg),
    "benchmark": df_summary.to_dict(orient="records"),
    "scma_snr_sweep": {"snr_db": snrs, "ber": bers, "ser": sers}
}
with open("results/metrics/colab_summary.json", "w") as f:
    json.dump(summary_data, f, indent=4)

print("Saved benchmark metrics to results/metrics/colab_benchmark_results.csv and colab_summary.json")

# In Google Colab, trigger file download if running in Colab environment:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("hybrid_noma_results", "zip", "results")
    files.download("hybrid_noma_results.zip")
    print("Download started for hybrid_noma_results.zip")
except ImportError:
    print("Local execution environment detected. Results preserved in 'results/' folder.")